# Tutorial 2: Accessing and modifying model data

This tutorial demonstrates how to get and set values of objects based on their topological key (tk).

## SIR 3S Installation

In [1]:
SIR3S_SIRGRAF_DIR = r"C:\3S\SIR 3S\SirGraf-90-15-00-24_Quebec-Upd2" #change to local path

## Imports

Note: The SIR 3S Toolkit requires the Sir3S_Toolkit.dll included in SIR 3S installations (version Quebec and higher).

In [2]:
import sir3stoolkit

The core of sir3stoolkit is a Python wrapper around basic functionality of SIR 3S, offering a low-level access to the creation, modification and simulation of SIR 3S models. In the future pure python subpackages may be added.

In [3]:
from sir3stoolkit.core import wrapper

In [4]:
sir3stoolkit

<module 'sir3stoolkit' from 'C:\\Users\\aUsername\\3S\\sir3stoolkit\\src\\sir3stoolkit\\__init__.py'>

The [wrapper package](https://3sconsult.github.io/sir3stoolkit/references/sir3stoolkit.core.html#sir3stoolkit.core.wrapper.Initialize_Toolkit) has to be initialized with reference to a SIR 3S (SIR Graf) installation.

In [5]:
wrapper.Initialize_Toolkit(SIR3S_SIRGRAF_DIR)

[2026-08-19 13:24:58,359] INFO in sir3stoolkit.core.wrapper: [Initialization] Using provided SirGraf path: C:\3S\SIR 3S\SirGraf-90-15-00-24_Quebec-Upd2
[2026-08-19 13:24:58,359] INFO in sir3stoolkit.core.wrapper: [Initialization] Using provided SirGraf path: C:\3S\SIR 3S\SirGraf-90-15-00-24_Quebec-Upd2


[2026-08-19 13:24:58,410] INFO in sir3stoolkit.core.wrapper: [Initialization] Initializing toolkit with SirGraf path: C:\3S\SIR 3S\SirGraf-90-15-00-24_Quebec-Upd2


## Initialization

The SIR 3S Toolkit contains two classes: [SIR3S_Model](https://3sconsult.github.io/sir3stoolkit/references/sir3stoolkit.core.html#sir3stoolkit.core.wrapper.SIR3S_Model) (model and data) and [SIR3S_View](https://3sconsult.github.io/sir3stoolkit/references/sir3stoolkit.core.html#sir3stoolkit.core.wrapper.SIR3S_View) (depiction in SIR Graf). All SIR 3S Toolkit functionality is accessed via the methods of these classes.

In [6]:
s3s = wrapper.SIR3S_Model()

[2026-08-19 13:25:01,049] INFO in sir3stoolkit.core.wrapper: [Model Class Initialization] Initialization complete


In [7]:
s3s_view = wrapper.SIR3S_View()

[2026-08-19 13:25:01,123] INFO in sir3stoolkit.core.wrapper: Initialization complete


## Open Model

In [8]:
dbFilePath=r"Toolkit_Tutorial2_Model.db3"

In [9]:
s3s.AllowSirMessageBox(bAllow=False) # If you encounter problems with a message box popping up when opening the model, this surpresses this

In [10]:
s3s.OpenModel(dbName=dbFilePath, 
              providerType=s3s.ProviderTypes.SQLite, 
              Mid="M-1-0-1", 
              saveCurrentlyOpenModel=False, 
              namedInstance="", 
              userID="", 
              password="")

[2026-08-19 13:25:13,564] INFO in sir3stoolkit.core.wrapper: Model is open for further operation


This model has been prepared and contains two nodes connected by a pipe.

## Get Values

Our goal is to find out what the type of the nodes is (PKON, QKON, etc.), and which length and roughness the pipe has.

The [GetValue()](https://3sconsult.github.io/sir3stoolkit/references/sir3stoolkit.core.html#sir3stoolkit.core.wrapper.SIR3S_Model.GetValue) function requires a tk to the object and the internal SIR 3S attribute name of the value we want to obtain. This guide will walk you through how to obtain those.

### SIR 3S object types

First, we need to obtain the internal SIR 3S object types to later pass to our function for obtaining the attribute names.

You can use dir() to get an overview over the different object types existing in SIR 3S.

In [11]:
object_types = [item for item in dir(s3s.ObjectTypes) if not (item.startswith('__') and item.endswith('__'))]
print(object_types)

['AGSN_HydraulicProfile', 'AirVessel', 'Arrow', 'Atmosphere', 'BlockConnectionNode', 'CalcPari', 'CharacteristicLossTable', 'CharacteristicLossTable_Row', 'Circle', 'Compressor', 'CompressorTable', 'CompressorTable_Row', 'ControlEngineeringNexus', 'ControlMode', 'ControlPointTable', 'ControlPointTable_Row', 'ControlValve', 'ControlVariableConverter', 'ControlVariableConverterRSTE', 'CrossSectionTable', 'CrossSectionTable_Row', 'DPGR_DPKT_DatapointDpgrConnection', 'DPGR_DataPointGroup', 'DPKT_Datapoint', 'DamageRatesTable', 'DamageRatesTable_Row', 'DeadTimeElement', 'Demand', 'DifferentialRegulator', 'DirectionalArrow', 'DistrictHeatingConsumer', 'DistrictHeatingFeeder', 'Divider', 'DriveEfficiencyTable', 'DriveEfficiencyTable_Row', 'DrivePowerTable', 'DrivePowerTable_Row', 'EBES_FeederGroups', 'EfficiencyConverterTable', 'EfficiencyConverterTable_Row', 'ElementQuery', 'EnergyRecoveryTable', 'EnergyRecoveryTable_Row', 'EnvironmentTemp', 'FWBZ_DistrictHeatingReferenceValues', 'FlapValve'

In our simple example we are only interested in the object types of nodes and pipes.

In [12]:
node_type=s3s.ObjectTypes.Node

In [13]:
pipe_type=s3s.ObjectTypes.Pipe

### GetTksofElementType()

Now, we obtain the tks of the nodes and pipes in this model.

We use [GetTksofElementType()](https://3sconsult.github.io/sir3stoolkit/references/sir3stoolkit.core.html#sir3stoolkit.core.wrapper.SIR3S_Model.GetTksofElementType) to create a list of all node tks in the model.

In [14]:
nodes=s3s.GetTksofElementType(ElementType=node_type)

In [15]:
print(nodes)

['4921762654790163024', '5483574590487309449', '5186991360372528632']


If you don't want to predefine your element type you can also just pass it like done below

In [16]:
pipes=s3s.GetTksofElementType(ElementType=s3s.ObjectTypes.Pipe)

In [17]:
print(pipes)

['4614473539164226343']


### GetTkFromIDReference()

We can use the [GetTkFromIDReference()](https://3sconsult.github.io/sir3stoolkit/references/sir3stoolkit.core.html#sir3stoolkit.core.wrapper.SIR3S_Model.GetGeometryInformation) function, in case we have an ID reference value of a certain element given and want to get its tk.

The nodes in our model have ID reference "A" and "B".

In [18]:
s3s.GetTkFromIDReference(IdRef="A", object_type=s3s.ObjectTypes.Node)

'4921762654790163024'

In [19]:
s3s.GetTkFromIDReference(IdRef="A", object_type=s3s.ObjectTypes.Node)

'4921762654790163024'

We can see that they are the same tks we got from GetTksofElementType().

### GetObjectTypeof_Key()

If we already have a tk/pk of an element we can use the [GetObjectTypeof_Key()](https://3sconsult.github.io/sir3stoolkit/references/sir3stoolkit.core.html#sir3stoolkit.core.wrapper.SIR3S_Model.GetObjectTypeof_Key) function to obtain its datatype.

In [20]:
print(s3s.GetObjectTypeof_Key(Key=nodes[0]))

ObjectTypes.Node


### GetPropertiesofElementType()

Now, we obtain the internal SIR 3S attribute names. These will be necessary for our value query.

We use [GetPropertiesofElementType()](https://3sconsult.github.io/sir3stoolkit/references/sir3stoolkit.core.html#sir3stoolkit.core.wrapper.SIR3S_Model.GetPropertiesofElementType) to create a list of all properties of nodes and pipes.

In [21]:
node_properties=s3s.GetPropertiesofElementType(ElementType=node_type)

In [22]:
print(node_properties)

['Name', 'Ktyp', 'Zkor', 'QmEin', 'Lfakt', 'Fkpzon', 'Fkfstf', 'Fkutmp', 'Fkfqps', 'Fkcont', 'Fk2lknot', 'Beschreibung', 'Idreferenz', 'Iplanung', 'Kvr', 'Qakt', 'Xkor', 'Ykor', 'NodeNamePosition', 'ShowNodeName', 'KvrKlartext', 'NumberOfVERB', 'HasBlockConnection', 'Tk', 'Pk', 'InVariant', 'GeometriesDiffer', 'SymbolFactor', 'bz.Drakonz', 'bz.Fk', 'bz.Fkpvar', 'bz.Fkqvar', 'bz.Fklfkt', 'bz.PhEin', 'bz.Tm', 'bz.Te', 'bz.PhMin']


In [23]:
pipe_properties=s3s.GetPropertiesofElementType(ElementType=pipe_type)

In [24]:
print(pipe_properties)

['Name', 'FkdtroRowd', 'Fkltgr', 'Fkstrasse', 'L', 'Lzu', 'Rau', 'Jlambs', 'Lambda0', 'Zein', 'Zaus', 'Zuml', 'Asoll', 'Indschall', 'Baujahr', 'Hal', 'Fkcont', 'Fk2lrohr', 'Beschreibung', 'Idreferenz', 'Iplanung', 'Kvr', 'LineWidthMM', 'DottedLine', 'DN', 'Di', 'KvrKlartext', 'HasClosedNSCHs', 'Tk', 'Pk', 'InVariant', 'Xkor', 'Ykor', 'GeometriesDiffer', 'bz.Fk', 'bz.Qsvb', 'bz.Irtrenn', 'bz.Leckstatus', 'bz.Leckstart', 'bz.Leckend', 'bz.Leckort', 'bz.Leckmenge', 'bz.Imptnz', 'bz.Zvlimptnz', 'bz.Kantenzv', 'bz.ITrennWithNSCH']


### GetValue()

Now, we can access values of indiviudal nodes with a corresponding tk.

We use [GetValue](https://3sconsult.github.io/sir3stoolkit/references/sir3stoolkit.core.html#sir3stoolkit.core.wrapper.SIR3S_Model.GetValue) for such individual value query.

#### Node 1

As tk we just use the first tk from our nodes list.

In [25]:
node1_ktyp=s3s.GetValue(Tk=nodes[0], propertyName='Ktyp')

In [26]:
print(node1_ktyp)

('PKON', 'string')


As you can see it returns a tuple value consisting of the actual value (here: 'PKON') of the attribute and the attribute data type.

You can access the indiviudal components as follows.

In [27]:
actual_value=node1_ktyp[0]

In [28]:
print(actual_value)

PKON


In [29]:
data_type=node1_ktyp[1]

In [30]:
print(data_type)

string


#### Node 2

Now we use the second tk from our nodes list.

In [31]:
node2_ktyp_actual_value=s3s.GetValue(Tk=nodes[1], propertyName='Ktyp')[0]

In [32]:
print(node2_ktyp_actual_value)

QKON


#### Pipe

For the pipe we will just ignore the datatypes of the attributes and just access the value.

In [33]:
print(pipe_properties)

['Name', 'FkdtroRowd', 'Fkltgr', 'Fkstrasse', 'L', 'Lzu', 'Rau', 'Jlambs', 'Lambda0', 'Zein', 'Zaus', 'Zuml', 'Asoll', 'Indschall', 'Baujahr', 'Hal', 'Fkcont', 'Fk2lrohr', 'Beschreibung', 'Idreferenz', 'Iplanung', 'Kvr', 'LineWidthMM', 'DottedLine', 'DN', 'Di', 'KvrKlartext', 'HasClosedNSCHs', 'Tk', 'Pk', 'InVariant', 'Xkor', 'Ykor', 'GeometriesDiffer', 'bz.Fk', 'bz.Qsvb', 'bz.Irtrenn', 'bz.Leckstatus', 'bz.Leckstart', 'bz.Leckend', 'bz.Leckort', 'bz.Leckmenge', 'bz.Imptnz', 'bz.Zvlimptnz', 'bz.Kantenzv', 'bz.ITrennWithNSCH']


In [34]:
pipe_length=s3s.GetValue(Tk=pipes[0], propertyName='L')[0]

In [35]:
print(pipe_length)

141,4214


In [36]:
pipe_roughness=s3s.GetValue(Tk=pipes[0], propertyName='RAU')[0]

In [37]:
print(pipe_roughness)

0,25


### GetGeometryInformation()

We can use the [GetGeometryInformation()](https://3sconsult.github.io/sir3stoolkit/references/sir3stoolkit.core.html#sir3stoolkit.core.wrapper.SIR3S_Model.GetGeometryInformation) function to obtain the technical geometries (Sachdatengeometrie) of the objects based on their tk.

In [38]:
for node in nodes:
    print(s3s.GetGeometryInformation(Tk=node))

POINT (0 0)
POINT (200 200)
POINT Z(401.84676647186279 110.7102632522583 0)


In [39]:
s3s.GetGeometryInformation(Tk=pipes[0])

'LINESTRING (0 0, 100 100)'

GetGeometryInformation() returns the technical data (Sachdatengeometrie) as a str in wkt format. The view geometry (Ansichtsgeometrie) may differ (check property "GeometriesDiffer").

In [40]:
s3s.GetValue(Tk=nodes[1], propertyName='GeometriesDiffer')

('True', 'boolean')

### GetGeometryData()

We can use the [GetGeometryData()](https://3sconsult.github.io/sir3stoolkit/modules.html#sir3stoolkit.core.wrapper.SIR3S_Model.GetGeometryData) function to obtain a more detailed information regarding the Geometry inlcluding the view geometry.

#### Node

In [41]:
geometry_data_node = s3s.GetGeometryData(elemTk = nodes[1])

In [42]:
geometry_data_node

(True, '{"X":200,"Y":200,"Z":0}', '{"X":200,"Y":100,"Z":0}', '', '', '', '')

We get a Tuple returned.

In [43]:
geometry_data_node[0] # is just a status check whether geometry was successfully retrieved

True

In [44]:
geometry_data_node[1] # represent the technical data (Sachdatengeometrie) of the element

'{"X":200,"Y":200,"Z":0}'

We can access it as a json object

In [45]:
import json

In [46]:
type(json.loads(geometry_data_node[1]))

dict

In [47]:
json.loads(geometry_data_node[1])["X"] # X, Y, Z coords can be accessed via a dict

200

In [48]:
json.loads(geometry_data_node[2]) # represent the view data (Ansichtsgeometrie) of the element, which differs from the technical data in this case

{'X': 200, 'Y': 100, 'Z': 0}

#### Pipe

In [49]:
s3s.GetValue(Tk=pipes[0], propertyName='GeometriesDiffer')

('True', 'boolean')

In [50]:
geometry_data_pipe = s3s.GetGeometryData(elemTk = pipes[0])

In [51]:
geometry_data_pipe

(True,
 '[{"X":0,"Y":0,"Z":0},{"X":100,"Y":100,"Z":0}]',
 '[{"X":0,"Y":0,"Z":0},{"X":200,"Y":100,"Z":0}]',
 '',
 '',
 '',
 '')

In [52]:
geometry_data_pipe_json_decoded = json.loads(geometry_data_pipe[1]) # again the technical geometry

In [53]:
type(geometry_data_pipe_json_decoded) # In this case we get a list of dicts and not a dict, since a pipe is defined by two points

list

The first element corresponds to the coords of the KI and the second to the KK node.

In [54]:
geometry_data_pipe_json_decoded[0]["X"]

0

In [55]:
geometry_data_pipe_json_decoded[1]["X"]

100

#### FlapValve (KLAP)

In [56]:
geometry_data_flap_valve = s3s.GetGeometryData(elemTk = s3s.GetTksofElementType(s3s.ObjectTypes.FlapValve)[0])

The elements with index 3, 4, 5, 6 of the geometry data tuple correspond to the connection lines between the element and the nodes connected to it.

3 - KI,
4 - KK,
5 - KI2,
6 - KK2,

In [57]:
geometry_data_flap_valve

(True,
 '{"X":312.12568283081055,"Y":105.92371225357056,"Z":0}',
 '',
 '[{"X":292.12568283081055,"Y":105.92371225357056,"Z":0},{"X":200,"Y":100,"Z":0}]',
 '[{"X":332.12568283081055,"Y":105.92371225357056,"Z":0},{"X":401.84676647186279,"Y":110.7102632522583,"Z":0}]',
 '',
 '')

In [58]:
geometry_data_flap_valve_conn_line_json_KI_decoded = json.loads(geometry_data_flap_valve[3]) # connection line between flap valve and KI node

In [59]:
type(geometry_data_flap_valve_conn_line_json_KI_decoded) # the line is represented the same way as the geometry of a pipe

list

In [60]:
geometry_data_flap_valve_conn_line_json_KI_decoded[0] # dict of coords element connection point

{'X': 292.12568283081055, 'Y': 105.92371225357056, 'Z': 0}

In [61]:
geometry_data_flap_valve_conn_line_json_KI_decoded[1] # dict of coords KI node

{'X': 200, 'Y': 100, 'Z': 0}

## Set Values

### SetValues()

You can use the [SetValue()](https://3sconsult.github.io/sir3stoolkit/references/sir3stoolkit.core.html#sir3stoolkit.core.wrapper.SIR3S_Model.SetValue) function to change non-result non-geometry values of objects based on their tk.

#### Node

In [62]:
print(s3s.GetValue(Tk=nodes[0], propertyName='Ktyp'))

('PKON', 'string')


Here we change a node from PKON to QKON

In [63]:
s3s.SetValue(nodes[0], propertyName='Ktyp', Value='QKON')

[2026-08-19 13:25:19,506] INFO in sir3stoolkit.core.wrapper: Value is set


In [64]:
print(s3s.GetValue(Tk=nodes[0], propertyName='Ktyp'))

('QKON', 'string')


#### Pipe

In [65]:
print(s3s.GetValue(Tk=pipes[0], propertyName='RAU')[0])

0,25


Here we change the roughness of a pipe from value 3 to 6.

In [66]:
s3s.SetValue(Tk=pipes[0], propertyName='RAU', Value='6')

[2026-08-19 13:25:19,812] INFO in sir3stoolkit.core.wrapper: Value is set


In [67]:
print(s3s.GetValue(Tk=pipes[0], propertyName='RAU')[0])

6


### SetGeometryInformation()

You can use the [SetGeometryInformation()](https://3sconsult.github.io/sir3stoolkit/references/sir3stoolkit.core.html#sir3stoolkit.core.wrapper.SIR3S_Model.SetGeometryInformation) function to change technical geometry (Sachdatengeometrie) values of objects based on their tk. The geometry has to be given in Wkt format.

Let's move the simple network 100 into positive x-direction.

#### Node

In [68]:
s3s.SetGeometryInformation(Tk=nodes[0], Wkt="POINT (100 0)")

[2026-08-19 13:25:20,013] INFO in sir3stoolkit.core.wrapper: Geometry Information is set correctly


True

In [69]:
s3s.SetGeometryInformation(Tk=nodes[1], Wkt="POINT (200 300)")

[2026-08-19 13:25:20,059] INFO in sir3stoolkit.core.wrapper: Geometry Information is set correctly


True

#### Pipe

In [70]:
s3s.SetGeometryInformation(Tk=pipes[0], Wkt="LINESTRING (100 0, 200 100)")

[2026-08-19 13:25:20,096] INFO in sir3stoolkit.core.wrapper: Geometry Information is set correctly


True

#### Setting regarding technical and view geometry

We can set whether the change of view geometry (Ansichtsgeometrie) also applies and automatically changes the technical geometry (Sachdatengeometrie).

In [71]:
tk_sirgraf = s3s.GetTksofElementType(s3s.ObjectTypes.SIRGRAF)[0]

In [72]:
s3s.SetValue(Tk=tk_sirgraf, propertyName="Upkc", Value="0") # no sync between view and technical

[2026-08-19 13:25:20,221] INFO in sir3stoolkit.core.wrapper: Value is set


In [ ]:
s3s.SetValue(Tk=tk_sirgraf, propertyName="Upkc", Value="1") # automatic sync between view and technical (only view change => technical change, but not the other way around)

[2026-08-19 13:25:20,264] INFO in sir3stoolkit.core.wrapper: Value is set


## Save Changes

If you want your changes made to the model to be saved use the [SaveChanges()](https://3sconsult.github.io/sir3stoolkit/references/sir3stoolkit.core.html#sir3stoolkit.core.wrapper.SIR3S_Model.SaveChanges) function.

In [74]:
#s3s.SaveChanges()

Alternatively, the user can save the changes when closing the model using the [CloseModel()](https://3sconsult.github.io/sir3stoolkit/references/sir3stoolkit.core.html#sir3stoolkit.core.wrapper.SIR3S_View.CloseModel) function.

In [75]:
#s3s.CloseModel(saveChangesBeforeClosing=True)

Now you are able to access model data from SIR 3S objects based on their tk. The above method only works for model data not for calculation results. 

__Next:__ Tutorial 3: Accessing simulation results